# Intelligent Crime Detection – Multimodal AI

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 14})
# To set the font size of the whole notebook

In [ ]:
pip install pandas numpy librosa matplotlib seaborn scikit-learn tensorflow

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 14})
# To set the font size of the whole notebook

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path
import os

# ── 1. Detect project root ──────────────────────────────────────────────────
current_dir = Path(os.getcwd()).resolve()

if current_dir.name == "notebook":
    PROJECT_ROOT = current_dir.parent
elif (current_dir / "notebook").exists():
    PROJECT_ROOT = current_dir
elif (current_dir / "README.md").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir
    for _ in range(5):
        if (PROJECT_ROOT / "README.md").exists():
            break
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Could not find project root (looking for README.md).")

# ── 2. Define all paths relative to PROJECT_ROOT ───────────────────────────
DATA_ROOT      = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR      = DATA_ROOT / "audio"
TEXT_DIR       = DATA_ROOT / "text"
AUDIO_METADATA = AUDIO_DIR / "UrbanSound8K.csv"
TEXT_METADATA  = TEXT_DIR / "Chicago_Police_Department_-_Illinois_Uniform_Crime_Reporting__IUCR__Codes.csv"
OUTPUT_DIR     = PROJECT_ROOT / "images"
MODEL_DIR      = PROJECT_ROOT / "models"
TEXT_DATA_PT   = PROJECT_ROOT / "data" / "text_data.pt"
MFCC_DATA_PT   = PROJECT_ROOT / "data" / "mfcc_data_with_labels_severity.pt"

# ── 3. Aliases used by downstream cells ────────────────────────────────────
base_path       = PROJECT_ROOT
audio_directory = AUDIO_DIR
audio_metadata  = AUDIO_METADATA
text_file_path  = TEXT_METADATA
output_dir      = OUTPUT_DIR

# ── 4. Diagnostics ─────────────────────────────────────────────────────────
print("Project root:  ", PROJECT_ROOT)
print("Audio dir:     ", audio_directory, "| exists?", audio_directory.exists())
print("Audio metadata:", audio_metadata,  "| exists?", audio_metadata.exists())
print("Text metadata: ", text_file_path,  "| exists?", text_file_path.exists())

if not TEXT_METADATA.exists():
    raise FileNotFoundError(f"Crime metadata not found: {TEXT_METADATA}")

In [ ]:
crime_data = pd.read_csv(TEXT_METADATA)   

print("\nFirst 5 rows of crime data:")
print(crime_data.head(5))

crime_data.head(10)

In [ ]:
crime_data.info()

In [ ]:
crime_data.isnull().sum()
# there are no null values

In [ ]:
# Assuming crime_data is already loaded
def process_text_data(crime_data):
    # Example: Count occurrences of each crime type
    crime_counts = crime_data['PRIMARY DESCRIPTION'].value_counts()
    return crime_counts

crime_counts = process_text_data(crime_data)
print(crime_counts)

In [ ]:
crime_data.duplicated().sum()
# there are duplicated values in the data but here duplicated values are possible since there is no primary key in the data

In [ ]:
print(crime_data['PRIMARY DESCRIPTION'].unique())

In [ ]:
crime_data[crime_data['ACTIVE']==True].count()

In [ ]:
crime_data[crime_data['ACTIVE']==True]

In [ ]:
crime_data[crime_data['ACTIVE']==False ]

In [ ]:
#True for Active Case
#False for Inactive Case

In [ ]:
# separate the columns into numerical and categorical columns
num_cols=[col for col in crime_data.columns if crime_data[col].dtype=='int64' ]
cat_cols=[col for col in crime_data.columns if col not in num_cols]

In [ ]:
# check some statistic of numerical and categorical columns separately
crime_data[cat_cols].describe()

In [ ]:
# performing chi square test to select important categorical features
from scipy.stats import chi2_contingency
def chi_square_test(crime_data, cat_cols, target_col):
    results = []
    for cat_col in cat_cols:
        contingency_table = pd.crosstab(crime_data[cat_col], crime_data[target_col])
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        results.append((cat_col, p))
    return results
chi_square_results = chi_square_test(crime_data, cat_cols[:-1], 'ACTIVE')
for result in chi_square_results:
    print(f"Chi-square p-value for {result[0]}: {result[1]}")

In [ ]:
print("Final audio metadata path we will use:", AUDIO_METADATA)
print("Does it really exist right now?", AUDIO_METADATA.exists())
print("Parent folder:", AUDIO_METADATA.parent)
print("Files in audio folder:", [f.name for f in AUDIO_METADATA.parent.glob("*")] if AUDIO_METADATA.parent.exists() else "Folder missing")


audio_md = pd.read_csv(AUDIO_METADATA)
print("\nFirst 5 rows of audio metadata:")
print(audio_md.head()) 

In [ ]:
audio_md.info()

In [ ]:
audio_md.isnull().sum()

In [ ]:
audio_md.duplicated().sum()

In [ ]:
print(audio_md['class'].unique())

In [ ]:
num_cols=[col for col in audio_md.columns if audio_md[col].dtype=='int64' ]
cat_cols=[col for col in audio_md.columns if col not in num_cols]


In [ ]:
audio_md[num_cols].describe()

In [ ]:
audio_md[cat_cols].describe()

In [ ]:
# Calculate duration
audio_md["duration"] = audio_md["end"] - audio_md["start"]

# Find the maximum duration
max_duration = audio_md["duration"].max()
print("Maximum Duration:", max_duration)

# Plot the durations
plt.figure(figsize=(10,5))
plt.hist(audio_md["duration"], bins=20, color='skyblue', edgecolor='black')
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Distribution of Audio File Durations")
plt.show()
plt.savefig(os.path.join(base_path, "images", "Distribution of Audio File Durations.png"), dpi=300, bbox_inches='tight', facecolor='white')


# MEL Frequency Cepstral Coefficients(MFCCs)

## MFCCs - compress and transform the spectrogram to mimic human hearing

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Load an audio file
audio_file_path = AUDIO_DIR / "fold1" / "143651-2-0-22.wav"

# Safety check (always do this)
if not audio_file_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_file_path}")

# Load audio
y, sr = librosa.load(audio_file_path, sr=None)

# Extract MFCCs
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

# Visualize MFCCs
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    mfccs,
    sr=sr,
    x_axis='time',
    y_axis='mel',
    cmap='coolwarm'
)
plt.colorbar(format='%+2.0f dB')
plt.title('MFCCs')
plt.tight_layout()
plt.savefig(os.path.join(base_path, "images", "MEL Frequency Cepstral Coefficients of Audio.png"), dpi=300, bbox_inches='tight', facecolor='white')

## Spectrograms: Used for Processing Audio Signals in time frequency space

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Load an audio file
audio_file_path = AUDIO_DIR / "fold1" / "143651-2-0-22.wav"

# Safety check (always do this)
if not audio_file_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_file_path}")

# Load audio
y, sr = librosa.load(audio_file_path, sr=None)

# Compute spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

# Plot spectrogram
plt.figure(figsize=(10, 4))
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title("Spectrogram of Crime Sound")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.savefig(os.path.join(base_path, "images", "Spectrogram of Crime Sound.png"), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy.signal import welch

# Load an audio file
audio_file_path = AUDIO_DIR / "fold4" / "7064-6-2-0.wav"

# Compute Spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

# Compute Power Spectral Density (PSD)
frequencies, psd = welch(y, sr, nperseg=1024)

# Plot Spectrogram & PSD
fig, ax = plt.subplots(2, 1, figsize=(10, 6))

# Plot PSD
ax[1].semilogy(frequencies, psd)
ax[1].set_title('Power Spectral Density (PSD)')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('Power')

# Plot Spectrogram
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', ax=ax[0])
ax[0].set_title('Spectrogram')
ax[0].set_xlabel('Time')
ax[0].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(os.path.join(base_path, "images", "Audio_Signal Picture.png"), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
!pip install librosa soundfile

In [ ]:
!pip install torch torchaudio transformers datasets


In [ ]:
from pathlib import Path

def get_audio_files(directory):
    """
    Find all .wav and .mp3 files recursively in the directory.
    Works with Path objects.
    """
    audio_files = []
    directory = Path(directory)  # make sure it's a Path
    
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")
    
    if not directory.is_dir():
        raise NotADirectoryError(f"Not a directory: {directory}")
    
    # Find all audio files (non-recursive for folds)
    for item in directory.iterdir():
        if item.is_dir() and item.name.startswith("fold"):
            for file in item.glob("*.[wW][aA][vV]"):  # case-insensitive .wav
                audio_files.append(str(file))
            for file in item.glob("*.[mM][pP][3]"):   # .mp3 if any
                audio_files.append(str(file))
    
    return sorted(audio_files)  # nice to have them sorted

In [ ]:
audio_files = get_audio_files(audio_directory)
print(f"Total audio files found: {len(audio_files)}")

if audio_files:
    print("First few files:")
    for f in audio_files[:5]:
        print(f"  - {f}")
else:
    print("No audio files found. Check:")
    print("  • Directory:", audio_directory)
    print("  • Exists?", audio_directory.exists())
    print("  • Subfolders:", [d.name for d in audio_directory.iterdir() if d.is_dir()])

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T


# Load the metadata CSV file
df = pd.read_csv(audio_metadata)

# Define severity rankings for audio events with clear documentation
AUDIO_SEVERITY_MAPPING = {
    "gun_shot": 2,        # High severity - immediate threat
    "siren": 2,          # High severity - emergency indication
    "car_horn": 1,       # Reassign to high severity
    "dog_bark": 1,       # Medium severity - disturbance
    "jackhammer": 1,     # Medium severity - construction hazard
    "drilling": 2,       # High severity - 
    "children_playing": 0,  # Low severity - normal activity
    "street_music": 0,    # Low severity - normal activity
    "engine_idling": 2,  # High severity - harmful emissions 
    "air_conditioner": 0  # Low severity - background noise
}

# Extract unique class labels and create a mapping
classes = df["class"].unique().tolist()
class_to_index = {cls: i for i, cls in enumerate(classes)}

def preprocess_audio_mfcc(file_path, n_mfcc=40, max_frames=100):
    try:
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            return None

        # Load and validate audio file
        waveform, sample_rate = torchaudio.load(file_path)
        if sample_rate < 8000:
            print(f"Low quality audio detected in {file_path}")
            return None

        # Convert to mono if stereo
        if waveform.shape[0] == 2:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Add small random noise for robustness
        noise = torch.randn_like(waveform) * 0.005
        waveform = waveform + noise

        # Extract MFCC features
        mfcc_transform = T.MFCC(sample_rate=sample_rate, n_mfcc=n_mfcc)
        mfcc = mfcc_transform(waveform)

        # Ensure consistent shape
        if mfcc.shape[-1] < max_frames:
            mfcc = torch.nn.functional.pad(mfcc, (0, max_frames - mfcc.shape[-1]))
        elif mfcc.shape[-1] > max_frames:
            mfcc = mfcc[:, :, :max_frames]

        return mfcc.squeeze(0)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None
        
def get_audio_mfcc_and_labels(directory, metadata_df):   # ← no max_files anymore
    audio_tensors, labels, severity_levels = [], [], []
    processed_count = 0
    skipped = 0

    for _, row in metadata_df.iterrows():
        fold_dir = f"fold{row['fold']}"
        file_name = row["slice_file_name"]
        class_label = row["class"]
        file_path = directory / fold_dir / file_name

        if file_path.exists():
            mfcc_tensor = preprocess_audio_mfcc(file_path)
            if mfcc_tensor is not None:
                audio_tensors.append(mfcc_tensor)
                labels.append(class_to_index[class_label])
                severity_levels.append(AUDIO_SEVERITY_MAPPING.get(class_label, 0))
                processed_count += 1
        else:
            skipped += 1

        if processed_count % 500 == 0 and processed_count > 0:
            print(f"Processed {processed_count} files... (skipped {skipped})")

    print(f"Final: Processed {processed_count} files, skipped {skipped}")
    if len(audio_tensors) == 0:
        raise RuntimeError( "ZERO audio files processed.\n"
            f"Last attempted path: {file_path}\n"
            "Check:\n"
            "1. Is directory correct? → " + str(directory) + "\n"
            "2. Does UrbanSound8K.csv have 'fold' and 'slice_file_name' columns?\n"
            "3. Are the fold1–fold10 folders present with .wav files?"
        )
    

    return (
        torch.stack(audio_tensors),
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(severity_levels, dtype=torch.long),
    )


    
# Process the dataset
print("Starting audio processing...")
mfcc_features, labels_tensor, severity_tensor = get_audio_mfcc_and_labels(audio_directory, df)

# Verify severity labels
print("Unique audio severity labels in final dataset:", torch.unique(severity_tensor))

# Save the processed dataset
save_path = os.path.join(base_path, "data", "mfcc_data_with_labels_severity.pt")
torch.save({
    "mfcc": mfcc_features, 
    "labels": labels_tensor, 
    "severity": severity_tensor
}, save_path)

# Print final shapes and confirmation
print("\nProcessing completed!")
print(f"Final MFCC batch shape: {mfcc_features.shape}")
print(f"Final labels shape: {labels_tensor.shape}")
print(f"Final severity shape: {severity_tensor.shape}")
print(f"Dataset saved to: {save_path}")

print(os.path.exists(os.path.join(base_path, "data", "mfcc_data_with_labels_severity.pt")))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
import os

# Configuration
OUTPUT_DIR = os.path.join(base_path, "images")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Severity mappings
TEXT_SEVERITY_MAP = {
    "THEFT": 1, "BATTERY": 2, "CRIMINAL DAMAGE": 1, "NARCOTICS": 2,
    "ASSAULT": 2, "BURGLARY": 2, "MOTOR VEHICLE THEFT": 2, "ROBBERY": 2,
    "DECEPTIVE PRACTICE": 1, "CRIMINAL TRESPASS": 1, "WEAPONS VIOLATION": 2,
    "OTHER OFFENSE": 0, "PUBLIC PEACE VIOLATION": 1, "OFFENSE INVOLVING CHILDREN": 2,
}

AUDIO_SEVERITY_MAP = {
    "gun_shot": 2, "siren": 2, "drilling": 2, "engine_idling": 2,
    "car_horn": 1, "dog_bark": 1, "jackhammer": 1,
    "children_playing": 0, "street_music": 0, "air_conditioner": 0
}

SEVERITY_LABELS = {0: "Low", 1: "Medium", 2: "High"}
SEVERITY_COLORS = {"Low": "#00d4aa", "Medium": "#ffb347", "High": "#ff6b6b"}


def plot_top_crimes(crime_data, top_n=10):
    """Bar plot of top N crime types"""
    crime_counts = crime_data["PRIMARY DESCRIPTION"].value_counts().head(top_n)
    
    plt.figure(figsize=(12, 6))
    bars = sns.barplot(x=crime_counts.values, y=crime_counts.index, 
                       palette="viridis", edgecolor='black')
    
    # Add value labels
    for i, count in enumerate(crime_counts.values):
        plt.text(count + 50, i, f'{count:,}', va='center', fontweight='bold')
    
    plt.title(f"Top {top_n} Crime Types", fontsize=16, fontweight='bold')
    plt.xlabel("Number of Incidents", fontsize=12)
    plt.ylabel("Crime Type", fontsize=12)
    plt.tight_layout()
    
    filepath = os.path.join(OUTPUT_DIR, 'top_crimes.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filepath}")


def plot_severity_distribution(data, severity_map, title, filename):
    """Generic severity distribution plot for text or audio data"""
    # Map to severity
    data["SEVERITY"] = data.iloc[:, 0].map(severity_map).fillna(0).astype(int)
    severity_counts = data["SEVERITY"].map(SEVERITY_LABELS).value_counts()
    
    # Ensure all severity levels are present
    for label in ["Low", "Medium", "High"]:
        if label not in severity_counts:
            severity_counts[label] = 0
    severity_counts = severity_counts[["Low", "Medium", "High"]]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(severity_counts.index, severity_counts.values, 
                   color=[SEVERITY_COLORS[x] for x in severity_counts.index],
                   edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 50,
                f'{int(height):,}', ha='center', va='bottom', 
                fontweight='bold', fontsize=12)
    
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel("Severity Level", fontsize=12, fontweight='bold')
    plt.ylabel("Number of Incidents", fontsize=12, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    filepath = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filepath}")


def plot_crime_wordcloud(crime_data):
    """Generate word cloud from crime descriptions"""
    text = " ".join(crime_data["PRIMARY DESCRIPTION"].dropna().str.lower())
    
    if not text.strip():
        print("⚠ Warning: No text data for word cloud")
        return None
    
    wordcloud = WordCloud(
        width=1200, height=600, 
        background_color='white', 
        colormap='viridis',
        max_words=100,
        relative_scaling=0.5
    ).generate(text)
    
    plt.figure(figsize=(14, 7))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title("Crime Description Word Cloud", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    
    filepath = os.path.join(OUTPUT_DIR, 'crime_wordcloud.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✓ Saved: {filepath}")
    return filepath


def plot_crime_pie_chart(crime_data, top_n=10):
    """Pie chart of top N crime types"""
    crime_counts = crime_data["PRIMARY DESCRIPTION"].value_counts().head(top_n)
    
    # Group remaining as "Others"
    total = crime_data["PRIMARY DESCRIPTION"].value_counts().sum()
    others = total - crime_counts.sum()
    if others > 0:
        crime_counts["Others"] = others
    
    plt.figure(figsize=(12, 8))
    colors = sns.color_palette("Set2", len(crime_counts))
    wedges, texts, autotexts = plt.pie(
        crime_counts.values, 
        labels=crime_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        textprops={'fontsize': 10, 'weight': 'bold'}
    )
    
    # Make percentage text white for better visibility
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(11)
    
    plt.title(f"Top {top_n} Crime Types Distribution", fontsize=16, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    
    filepath = os.path.join(OUTPUT_DIR, 'crime_pie_chart.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filepath}")
    return filepath


def run_all_visualizations(crime_data, audio_data):
    """Generate all visualizations"""
    print("\n" + "="*60)
    print("GENERATING CRIME DATA VISUALIZATIONS")
    print("="*60 + "\n")
    
    # 1. Top crimes bar plot
    plot_top_crimes(crime_data, top_n=10)
    
    # 2. Text severity distribution
    plot_severity_distribution(
        crime_data[["PRIMARY DESCRIPTION"]].copy(),
        TEXT_SEVERITY_MAP,
        "Crime Severity Distribution (Text Data)",
        "severity_text.png"
    )
    
    # 3. Audio severity distribution
    plot_severity_distribution(
        audio_data[["class"]].copy(),
        AUDIO_SEVERITY_MAP,
        "Audio Severity Distribution",
        "severity_audio.png"
    )
    
    # 4. Pie chart
    plot_crime_pie_chart(crime_data, top_n=10)
    
    # 5. Word cloud
    plot_crime_wordcloud(crime_data)
    
    print("\n" + "="*60)
    print(f"✓ ALL VISUALIZATIONS SAVED TO: {OUTPUT_DIR}/")
    print("="*60)


# Example usage
if __name__ == "__main__":
    # Load your data (USE VARIABLES, NOT STRINGS)
    crime_data = pd.read_csv(text_file_path)
    audio_data = pd.read_csv(audio_metadata)

    # Run visualizations
    run_all_visualizations(crime_data, audio_data)
    
    print("Import this module and call run_all_visualizations(crime_data, audio_data)")

In [ ]:
from transformers import DistilBertTokenizer  

# Load Crime Data
crime_data = pd.read_csv(text_file_path)

# Define Severity Categories
TEXT_SEVERITY_MAPPING = {
    "HOMICIDE": 2,
    "CRIMINAL SEXUAL ASSAULT": 2,
    "ROBBERY": 2,
    "BATTERY": 2,
    "ASSAULT": 2,
    "STALKING": 2,
    "BURGLARY": 2,
    "MOTOR VEHICLE THEFT": 2,
    "ARSON": 2,
    "HUMAN TRAFFICKING": 2,
    "KIDNAPPING": 2,
    "WEAPONS VIOLATION": 2,
    "DECEPTIVE PRACTICE": 1,
    "CRIMINAL DAMAGE": 1,
    "CRIMINAL TRESPASS": 1,
    "CONCEALED CARRY LICENSE VIOLATION": 1,
    "PROSTITUTION": 1,
    "OFFENSE INVOLVING CHILDREN": 1,
    "SEX OFFENSE": 1,
    "GAMBLING": 1,
    "NARCOTICS": 1,
    "OTHER NARCOTIC VIOLATION": 1,
    "LIQUOR LAW VIOLATION": 1,
    "CRIMINAL ABORTION": 1,
    "INTERFERENCE WITH PUBLIC OFFICER": 1,
    "INTIMIDATION": 1,
    "PUBLIC PEACE VIOLATION": 0,
    "RITUALISM": 0,
    "NON-CRIMINAL": 0,
    "OBSCENITY": 0,
    "PUBLIC INDECENCY": 0,
    "OTHER OFFENSE": 0
}

# Assign Severity Labels
crime_data["SEVERITY"] = crime_data["PRIMARY DESCRIPTION"].map(TEXT_SEVERITY_MAPPING).fillna(0).astype(int)

# Tokenization
# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize text descriptions
tokenized_output = tokenizer(
    list(crime_data["PRIMARY DESCRIPTION"]), 
    padding=True,          
    truncation=True,       
    return_tensors="pt"    
)

# Extract tokenized input tensor
text_tokens = tokenized_output["input_ids"]

# Convert severity labels to numeric values
text_labels = torch.tensor(crime_data["SEVERITY"].values)
text_severity = torch.tensor(crime_data["SEVERITY"].values)

# Save tokenized data
torch.save({
    "text_tokens": text_tokens, 
    "labels": text_labels, 
    "severity": text_severity
}, os.path.join(base_path, "data", "text_data.pt"))

# Load tokenized data
text_data = torch.load(os.path.join(base_path, "data", "text_data.pt"))
text_features = text_data["text_tokens"]
text_labels = text_data["labels"]
text_severity = text_data["severity"]

print("Text_features:", text_features.shape)
print("Text_labels:", text_labels.shape)
print(f"Tokenized text and labels saved in {base_path}.")

# Select a sample text from the dataset
sample_text = crime_data["PRIMARY DESCRIPTION"].iloc[0]  # First crime description

# Tokenize the sample text
sample_output = tokenizer(sample_text, padding=True, truncation=True, return_tensors="pt")

# Extract sample tokens and token IDs
sample_tokens = tokenizer.convert_ids_to_tokens(sample_output["input_ids"].squeeze().tolist())
sample_ids = sample_output["input_ids"].squeeze().tolist()


vocab_size = tokenizer.vocab_size
special_tokens = tokenizer.special_tokens_map

max_length = text_tokens.size(1)  # Maximum sequence length after padding
total_tokens = text_tokens.numel()  # Tota number of tokens including padding
unique_tokens = len(torch.unique(text_tokens))

features = tokenized_output.keys()
sequence_lengths = (text_tokens != tokenizer.pad_token_id).sum(dim=1)
avg_sequence_length = sequence_lengths.float().mean()

print("\nNLP Analysis:")
print(f"Sample text: '{sample_text}'")
print(f"Sample tokens: {sample_tokens}")
print(f"Sample token IDs: {sample_ids}")
print(f"\nVocabulary size: {vocab_size}")
print(f"Special tokens: {special_tokens}")
print(f"\nMaximum sequence length: {max_length}")
print(f"Average sequence length: {avg_sequence_length:.2f}")
print(f"Total tokens in dataset: {total_tokens}")
print(f"Unique tokens used: {unique_tokens}")
print(f"\nFeatures available: {list(features)}")
print("Feature dimensions:", {k: v.shape for k, v in tokenized_output.items()})

print(os.path.exists(os.path.join(base_path, "text_data.pt")))
print(crime_data["SEVERITY"].unique())
#print("Unique audio severity labels:", torch.unique(audio_severity))
print("Unique text severity labels:", torch.unique(text_severity))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AudioEncoder(nn.Module):
    def __init__(self, input_dim=40, hidden_dim=128):
        super(AudioEncoder, self).__init__()
        
        # CNN part - feature extraction from MFCC
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(64)
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(128)
        
        self.pool  = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        
        # LSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.out_dim = hidden_dim * 2  # 256 if hidden_dim=128
        
    def forward(self, x):
    # Safety: make sure channels are second dimension
        if x.shape[1] != 40:  # if time is second
            print("Warning: Transposing MFCC input to [B, channels=40, time]")
            x = x.transpose(1, 2)
    
    # Now x is [B, 40, T]
        x = F.relu(self.bn1(self.conv1(x)))     # conv1 expects channels=40
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
    
        x = x.transpose(1, 2)                   # [B, T/2, 128] for LSTM
        lstm_out, _ = self.lstm(x)
        features = lstm_out.mean(dim=1)
        features = self.layer_norm(features)
    
        return features




In [ ]:
from transformers import DistilBertModel

class TextEncoder(nn.Module):
    def __init__(self):
        super(TextEncoder, self).__init__()
        print("Loading DistilBERT model...")
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.out_dim = 768  # hidden size of DistilBERT

        # Optional: freeze at init (you can unfreeze later)
        # for p in self.distilbert.parameters():
        #     p.requires_grad = False

    def forward(self, input_ids):
        # input_ids: [batch, seq_len] – long type
        attention_mask = (input_ids != 0).long()   # 0 = padding token
        
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Mean pooling over sequence length
        last_hidden = outputs.last_hidden_state     # [B, seq, 768]
        pooled = last_hidden.mean(dim=1)            # [B, 768]
        
        return pooled

In [ ]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, embed_dim=256, num_classes=3):
        super().__init__()
        
        self.audio_encoder = AudioEncoder()
        self.text_encoder  = TextEncoder()
        
        # Project both to the same dimension
        self.audio_proj = nn.Linear(self.audio_encoder.out_dim, embed_dim)
        self.text_proj  = nn.Linear(self.text_encoder.out_dim, embed_dim)
        
        # Final classifier
        self.classifier = nn.Linear(embed_dim * 2, num_classes)
        
        self.embed_dim = embed_dim
        self.num_classes = num_classes

    def forward(self, audio, text):
        # audio: [B, 40, time]
        # text:  [B, seq_len] token ids
        
        audio_feat = self.audio_encoder(audio)          # [B, 256]
        text_feat  = self.text_encoder(text)            # [B, 768]
        
        audio_emb = F.relu(self.audio_proj(audio_feat)) # [B, 256]
        text_emb  = F.relu(self.text_proj(text_feat))   # [B, 256]
        
        fused = torch.cat([audio_emb, text_emb], dim=1) # [B, 512]
        logits = self.classifier(fused)                 # [B, 3]
        
        return logits, audio_emb, text_emb

    @staticmethod
    def final_severity(audio_pred: int, text_pred: int) -> int:
        """
        Conservative rule:
        - High (2) if ANY modality says High
        - Medium (1) if any says Medium and none High
        - Low (0) otherwise
        """
        if audio_pred == 2 or text_pred == 2:
            return 2
        if audio_pred == 1 or text_pred == 1:
            return 1
        return 0

In [ ]:
def supervised_contrastive_loss(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 0.25
) -> torch.Tensor:
    """
    NaN-resistant version
    """
    embeddings = F.normalize(embeddings, dim=-1)
    sim = torch.mm(embeddings, embeddings.t()) / temperature
    
    # Mask for positives (same label), exclude self
    mask = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    mask.fill_diagonal_(0)
    
    # Numerical stability
    exp_sim = torch.exp(sim.clamp(max=80))  # prevent exp explosion
    
    positives = exp_sim * mask
    num = positives.sum(dim=1, keepdim=True)
    
    den = exp_sim.sum(dim=1, keepdim=True)
    den = torch.clamp(den - torch.exp(sim.diag()).unsqueeze(1), min=1e-6)
    
    pos_count = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    
    loss = -torch.log(num / den + 1e-8) / pos_count
    return loss.mean()

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# 1. Paths (change if your folder is different)
base_path = PROJECT_ROOT

mfcc_path = os.path.join(base_path, "data","mfcc_data_with_labels_severity.pt")
text_path = os.path.join(base_path, "data",  "text_data.pt")

# 2. Load — everything stays on CPU
print("Loading preprocessed data (CPU only)...")
mfcc_dict = torch.load(mfcc_path, map_location='cpu')
text_dict = torch.load(text_path, map_location='cpu')

mfcc_features   = mfcc_dict["mfcc"].float()           # [4612, 40, 100]
audio_severity  = mfcc_dict["severity"].long()        # [4612]

text_tokens     = text_dict["text_tokens"].long()     # MUST BE LONG for BERT
text_severity   = text_dict["severity"].long()        # [4612]

print(f"Loaded {len(mfcc_features)} samples")
print(f"MFCC shape:     {mfcc_features.shape}")
print(f"Text tokens:    {text_tokens.shape} (dtype: {text_tokens.dtype})")
print(f"Audio severity: {torch.unique(audio_severity)}")
print(f"Text severity:  {torch.unique(text_severity)}")

# 3. Safe split using MAX severity for stratification
print("\nSplitting dataset...")
max_severity = torch.max(audio_severity, text_severity)

indices = list(range(len(mfcc_features)))
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    random_state=42,
    stratify=max_severity.numpy()
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=42,
    stratify=max_severity[temp_idx].numpy()
)

# 4. Create datasets (CPU only)
def create_dataset(idx_list):
    return TensorDataset(
        mfcc_features[idx_list],
        text_tokens[idx_list],
        torch.zeros(len(idx_list), 1),               # dummy audio label
        torch.zeros(len(idx_list), 1),               # dummy text label
        audio_severity[idx_list].unsqueeze(1),
        text_severity[idx_list].unsqueeze(1)
    )

train_dataset = create_dataset(train_idx)
val_dataset   = create_dataset(val_idx)
test_dataset  = create_dataset(test_idx)

# 5. DataLoaders — small batch, no pin_memory, num_workers=0 (Windows-safe)
batch_size = 16   # Start small to avoid OOM

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# 6. Print confirmation
print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset):5d} ({len(train_dataset)/4612*100:.1f}%)")
print(f"  Val:   {len(val_dataset):5d} ({len(val_dataset)/4612*100:.1f}%)")
print(f"  Test:  {len(test_dataset):5d} ({len(test_dataset)/4612*100:.1f}%)")

print("\nReady for training! No data moved to GPU yet.")

In [ ]:
# Create model & move ONLY model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MultimodalFusionModel(embed_dim=256, num_classes=3)
model = model.to(device)

# Freeze DistilBERT initially
for param in model.text_encoder.distilbert.parameters():
    param.requires_grad = False

print(f"Model ready. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

def train_model(model, train_loader, val_loader, epochs=20):  # Start with 15 – increase later
    device = next(model.parameters()).device
    
    # Class weights based on MAX severity (more robust)
    print("Computing class weights from max severity...")
    all_targets = []
    for batch in train_loader:
        sev_a, sev_t = batch[4], batch[5]
        target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()
        all_targets.append(target)
    all_targets = torch.cat(all_targets)
    class_counts = torch.bincount(all_targets, minlength=3)
    weights = 1.0 / (class_counts.float() + 1.0)  # +1 prevents infinity
    weights = weights / weights.sum()
    print(f"Class weights (Low/Med/High): {weights.tolist()}")
    
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = optim.AdamW(
        model.parameters(),
        lr=5e-5,              # Lower LR for stability on CPU
        weight_decay=1e-4
    )
    
    # Cosine annealing with warmup-like behavior (good for BERT-like models)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    best_val_f1 = -1.0
    patience = 6
    patience_counter = 0
    
    for epoch in range(epochs):
        # Unfreeze last 2 layers after epoch 5 (gradual fine-tuning)
        if epoch == 5:
            print("→ Unfreezing last 2 DistilBERT layers for fine-tuning")
            for param in model.text_encoder.distilbert.transformer.layer[-2:].parameters():
                param.requires_grad = True
            # Re-create optimizer to include new params
            optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs - epoch, eta_min=1e-6)
        
        model.train()
        train_loss = train_correct = train_total = 0
        train_preds, train_targets_list = [], []
        
        for batch in train_loader:
            audio, text, _, _, sev_a, sev_t = [b.to(device) for b in batch]
            target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()
            
            optimizer.zero_grad()
            
            logits, audio_emb, text_emb = model(audio, text)
            
            ce_loss = criterion(logits, target)
            
            # Contrastive loss — only after epoch 3, and low weight
            if epoch >= 3:
                audio_scl = supervised_contrastive_loss(audio_emb, target, temperature=0.25)
                text_scl  = supervised_contrastive_loss(text_emb, target, temperature=0.25)
                scl_loss = (audio_scl + text_scl) / 2
                total_loss = ce_loss + 0.20 * scl_loss  # 20% weight – conservative
            else:
                total_loss = ce_loss
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # Prevent explosion
            optimizer.step()
            
            _, predicted = logits.max(dim=1)
            train_loss += total_loss.item()
            train_correct += predicted.eq(target).sum().item()
            train_total += target.size(0)
            
            train_preds.extend(predicted.cpu().numpy())
            train_targets_list.extend(target.cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        train_f1 = f1_score(train_targets_list, train_preds, average='weighted')
        
        # Validation
        model.eval()
        val_loss = val_correct = val_total = 0
        val_preds, val_targets_list = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                audio, text, _, _, sev_a, sev_t = [b.to(device) for b in batch]
                target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()
                
                logits, _, _ = model(audio, text)
                loss = criterion(logits, target)
                
                val_loss += loss.item()
                _, predicted = logits.max(dim=1)
                val_correct += predicted.eq(target).sum().item()
                val_total += target.size(0)
                
                val_preds.extend(predicted.cpu().numpy())
                val_targets_list.extend(target.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total
        val_f1 = f1_score(val_targets_list, val_preds, average='weighted')
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train: {train_loss:.4f} | Acc: {train_acc:.2f}% | F1: {train_f1:.4f}")
        print(f"  Val:   {val_loss:.4f} | Acc: {val_acc:.2f}% | F1: {val_f1:.4f}")
        
        scheduler.step()
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(base_path, "models", "best_model.pt"))
            print("   → Saved best model (improved val F1)")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Plot training curves
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Accuracy (%)')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history['train_f1'], label='Train F1')
    plt.plot(history['val_f1'], label='Val F1')
    plt.title('Weighted F1 Score')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, "images", "training_history.png"))
    plt.show()
    
    return model, history

# ==============================================
# Run Training
# ==============================================
print("\nStarting training...")
print("Note: On CPU this may take 5–20 min per epoch. Be patient.")
model, history = train_model(model, train_loader, val_loader, epochs=20)

print("\nTraining finished. Best model saved as 'best_model.pt'")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# ────────────────────────────────────────────────
# Evaluation Setup
# ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model (already done in your training, but just in case)
model.load_state_dict(torch.load(os.path.join(base_path, "models", "best_model.pt")))
model = model.to(device)
model.eval()

print("Evaluating on test set...")

all_fused_preds = []
all_final_preds = []
all_targets = []
all_probs = []          # for ROC (from fused logits)
total_loss = 0.0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for batch in test_loader:
        audio, text, _, _, sev_a, sev_t = [b.to(device) for b in batch]
        text = text.long()
        
        # Ground truth target (max severity)
        target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()
        
        # Forward pass: fused model
        logits, audio_emb, text_emb = model(audio, text)
        
        loss = criterion(logits, target)
        total_loss += loss.item()
        
        probs = F.softmax(logits, dim=1)
        fused_pred = logits.argmax(dim=1)
        
        # ── Unimodal predictions for conservative final_severity ──
        # Audio-only
        audio_feat = model.audio_encoder(audio)
        audio_emb_proj = F.relu(model.audio_proj(audio_feat))
        fake_text = torch.zeros_like(audio_emb_proj)
        audio_logits = model.classifier(torch.cat([audio_emb_proj, fake_text], dim=1))
        audio_pred = audio_logits.argmax(dim=1)
        
        # Text-only
        text_feat = model.text_encoder(text)
        text_emb_proj = F.relu(model.text_proj(text_feat))
        fake_audio = torch.zeros_like(text_emb_proj)
        text_logits = model.classifier(torch.cat([fake_audio, text_emb_proj], dim=1))
        text_pred = text_logits.argmax(dim=1)
        
        # Conservative rule: any High → High, etc.
        batch_final = [
            model.final_severity(a.item(), t.item())
            for a, t in zip(audio_pred, text_pred)
        ]
        
        # Collect everything
        all_fused_preds.extend(fused_pred.cpu().numpy())
        all_final_preds.extend(batch_final)
        all_targets.extend(target.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# ────────────────────────────────────────────────
# Compute Metrics
# ────────────────────────────────────────────────
avg_loss = total_loss / len(test_loader)

fused_acc = 100. * np.mean(np.array(all_fused_preds) == np.array(all_targets))
final_acc = 100. * np.mean(np.array(all_final_preds) == np.array(all_targets))
final_f1  = f1_score(all_targets, all_final_preds, average='weighted')

print(f"\nTest Loss (fused): {avg_loss:.4f}")
print(f"Fused Model Accuracy:     {fused_acc:.2f}%")
print(f"Final Conservative Acc:   {final_acc:.2f}%")
print(f"Final Weighted F1-Score:  {final_f1:.4f}\n")

print("Classification Report (Final Conservative Predictions):")
print(classification_report(all_targets, all_final_preds,
                           target_names=["Low", "Medium", "High"], digits=3))

# ────────────────────────────────────────────────
# Confusion Matrix - FIXED 
# ────────────────────────────────────────────────
cm = confusion_matrix(all_targets, all_final_preds)
print("\nConfusion Matrix (rows=True, cols=Predicted):")
print(cm)


# ═══════════════════════════════════════════════════════════════
# METHOD 3: Dual-tone text (black on light cells, white on dark)
# ═══════════════════════════════════════════════════════════════
plt.figure(figsize=(10, 8))

# First create the heatmap without annotations
ax = sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                 xticklabels=["Low", "Medium", "High"],
                 yticklabels=["Low", "Medium", "High"],
                 linewidths=2, linecolor='black',
                 vmin=0, vmax=cm.max(),
                 cbar_kws={'label': 'Count'})

# Add annotations with adaptive text color
threshold = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        text_color = "white" if cm[i, j] > threshold else "black"
        text = ax.text(j + 0.5, i + 0.5, str(cm[i, j]),
                      ha="center", va="center",
                      color=text_color, fontsize=20, fontweight='bold')

plt.xlabel('Predicted Severity', fontsize=14, fontweight='bold')
plt.ylabel('True Severity', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix - Final Conservative Fusion\n(Adaptive Text Color)', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(base_path, "images",'confusion_matrix_final_method3.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ────────────────────────────────────────────────
# ROC Curves (using fused probabilities)
# ────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
colors = ['#00d4aa', '#ffb347', '#ff6b6b']  # Matching dashboard colors
for i, name in enumerate(["Low", "Medium", "High"]):
    y_true_bin = np.array(all_targets) == i
    y_score = np.array(all_probs)[:, i]
    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=3, label=f'{name} (AUC = {roc_auc:.3f})', color=colors[i])

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
plt.title('ROC Curves - One-vs-Rest (Fused Model)', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(base_path, "images",'roc_curves.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ────────────────────────────────────────────────
# Final Summary
# ────────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL EVALUATION SUMMARY")
print("="*70)
print(f"  • Fused Accuracy:              {fused_acc:.2f}%")
print(f"  • Conservative Accuracy:       {final_acc:.2f}%  ← Use this in deployment")
print(f"  • Weighted F1 (conservative):  {final_f1:.4f}")
print(f"  • Test Loss:                   {avg_loss:.4f}")
print("\n  • Plot saved:")
print("    - confusion_matrix_final_method3.png (Adaptive text color)")
print("    - roc_curves.png")
print("="*70)

# Print per-class metrics
print("\n" + "="*70)
print("PER-CLASS METRICS")
print("="*70)
class_names = ["Low", "Medium", "High"]
for i, name in enumerate(class_names):
    # Precision, Recall, F1
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n{name} Severity:")
    print(f"  Samples:    {cm[i, :].sum()}")
    print(f"  Correct:    {tp}")
    print(f"  Precision:  {precision:.3f}")
    print(f"  Recall:     {recall:.3f}")
    print(f"  F1-Score:   {f1:.3f}")

print("="*70)

In [ ]:
from sklearn.metrics import fbeta_score, classification_report
import torch
import torch.nn.functional as F
import numpy as np

def evaluate_model_with_f2(model, test_loader, class_names=["Low", "Medium", "High"]):
    model.eval()
    device = next(model.parameters()).device

    all_fused_preds = []
    all_final_preds = []
    all_targets = []
    all_probs = []
    total_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in test_loader:
            audio, text, _, _, sev_a, sev_t = [b.to(device) for b in batch]
            text = text.long()
            target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()

            logits, audio_emb, text_emb = model(audio, text)
            loss = criterion(logits, target)
            total_loss += loss.item()

            probs = F.softmax(logits, dim=1)
            fused_pred = logits.argmax(dim=1)

            # Conservative unimodal preds
            audio_feat = model.audio_encoder(audio)
            audio_emb_proj = F.relu(model.audio_proj(audio_feat))
            zero_text = torch.zeros_like(audio_emb_proj)
            audio_logits = model.classifier(torch.cat([audio_emb_proj, zero_text], dim=1))
            audio_pred = audio_logits.argmax(dim=1)

            text_feat = model.text_encoder(text)
            text_emb_proj = F.relu(model.text_proj(text_feat))
            zero_audio = torch.zeros_like(text_emb_proj)
            text_logits = model.classifier(torch.cat([zero_audio, text_emb_proj], dim=1))
            text_pred = text_logits.argmax(dim=1)

            batch_final = [model.final_severity(a.item(), t.item())
                           for a, t in zip(audio_pred, text_pred)]

            all_fused_preds.extend(fused_pred.cpu().numpy())
            all_final_preds.extend(batch_final)
            all_targets.extend(target.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(test_loader)

    # Metrics
    fused_acc = 100. * np.mean(np.array(all_fused_preds) == np.array(all_targets))
    final_acc = 100. * np.mean(np.array(all_final_preds) == np.array(all_targets))
    final_f1 = f1_score(all_targets, all_final_preds, average='weighted')
    final_f2 = fbeta_score(all_targets, all_final_preds, average='weighted', beta=2)  # F2 emphasizes recall

    print(f"Test Loss (fused): {avg_loss:.4f}")
    print(f"Fused Accuracy: {fused_acc:.2f}%")
    print(f"Final Conservative Accuracy: {final_acc:.2f}%")
    print(f"Final Weighted F1: {final_f1:.4f}")
    print(f"Final F2-Score (beta=2): {final_f2:.4f}\n")

    print("Classification Report (Final Conservative):")
    print(classification_report(all_targets, all_final_preds,
                                target_names=class_names, digits=3))

    # Return all for further analysis
    return {
        'loss': avg_loss,
        'fused_acc': fused_acc,
        'final_acc': final_acc,
        'final_f1': final_f1,
        'final_f2': final_f2,
        'targets': all_targets,
        'final_preds': all_final_preds,
        'probs': np.array(all_probs)
    }

# Run it
print("Running F2-focused evaluation...\n")
test_results = evaluate_model_with_f2(model, test_loader)
print("\nEvaluation Summary:")
print(f" • Loss:         {test_results['loss']:.4f}")
print(f" • Final Acc:    {test_results['final_acc']:.2f}%")
print(f" • Final F1:     {test_results['final_f1']:.4f}")
print(f" • Final F2:     {test_results['final_f2']:.4f}")

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import fbeta_score

def run_ablation_study(model, test_loader):
    """Safe ablation: full fusion vs audio-only vs text-only"""
    results = {}
    device = next(model.parameters()).device
    model.eval()

    def evaluate_subset(get_logits_fn):
        """Run full evaluation loop with custom logit getter"""
        all_preds = []
        all_targets = []
        total_loss = 0.0
        criterion = torch.nn.CrossEntropyLoss()

        with torch.no_grad():
            for batch in test_loader:
                audio, text, _, _, sev_a, sev_t = [b.to(device) for b in batch]
                text = text.long()
                target = torch.max(sev_a.squeeze(1), sev_t.squeeze(1)).long()

                # Get logits using custom function
                logits = get_logits_fn(audio, text)

                loss = criterion(logits, target)
                total_loss += loss.item()

                _, pred = logits.max(dim=1)
                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(target.cpu().numpy())

        acc = 100. * sum(p == t for p, t in zip(all_preds, all_targets)) / len(all_targets)
        f2 = fbeta_score(all_targets, all_preds, average='weighted', beta=2)
        avg_loss = total_loss / len(test_loader)

        return {'accuracy': acc, 'f2_score': f2, 'loss': avg_loss}

    print("Running ablation study...\n")

    # 1. Full model (normal forward)
    def full_logits(audio, text):
        return model(audio, text)[0]  # only logits

    print("→ Full fusion (audio + text)")
    results['full_fusion'] = evaluate_subset(full_logits)

    # 2. Audio-only
    def audio_only_logits(audio, text):
        audio_feat = model.audio_encoder(audio)
        audio_emb = F.relu(model.audio_proj(audio_feat))
        zero_text = torch.zeros_like(audio_emb).to(audio_feat.device)
        fused = torch.cat([audio_emb, zero_text], dim=1)
        return model.classifier(fused)

    print("→ Audio-only")
    results['audio_only'] = evaluate_subset(audio_only_logits)

    # 3. Text-only
    def text_only_logits(audio, text):
        text_feat = model.text_encoder(text)
        text_emb = F.relu(model.text_proj(text_feat))
        zero_audio = torch.zeros_like(text_emb).to(text_feat.device)
        fused = torch.cat([zero_audio, text_emb], dim=1)
        return model.classifier(fused)

    print("→ Text-only")
    results['text_only'] = evaluate_subset(text_only_logits)

    # ── Summary Table ──────────────────────────────────────────────
    print("\nAblation Study Results:")
    print("-" * 70)
    print(f"{'Config':<20} {'Accuracy (%)':<15} {'F2-Score':<12} {'Loss':<12}")
    print("-" * 70)

    for config, metrics in results.items():
        print(f"{config:<20} {metrics['accuracy']:<15.2f} {metrics['f2_score']:<12.4f} {metrics['loss']:<12.4f}")

    print("-" * 70)

    # Modality contribution insight
    audio_acc = results['audio_only']['accuracy']
    text_acc  = results['text_only']['accuracy']
    full_acc  = results['full_fusion']['accuracy']

    print(f"\nModality Contribution:")
    print(f"  • Audio-only:   {audio_acc:.2f}%")
    print(f"  • Text-only:    {text_acc:.2f}%")
    print(f"  • Full fusion:  {full_acc:.2f}%")

    if full_acc > max(audio_acc, text_acc) + 1.0:
        print("  → Fusion provides clear multimodal benefit")
    elif full_acc >= max(audio_acc, text_acc):
        print("  → Fusion is comparable to best unimodal")
    else:
        print("  → Fusion underperforms → possible domain mismatch or alignment issue")

    return results


# ────────────────────────────────────────────────
# Run it
# ────────────────────────────────────────────────
print("Starting ablation study...\n")
ablation_results = run_ablation_study(model, test_loader)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap

def visualize_ablation_results(ablation_results):
    """Visualize ablation study results with bar chart"""
    
    if not ablation_results:
        print("No ablation results available to visualize.")
        return
    
    # Convert to DataFrame
    df = pd.DataFrame.from_dict(ablation_results, orient='index')
    
    # Sort by F2-Score descending (best on left)
    df = df.sort_values('f2_score', ascending=False)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Safe style fallback
    try:
        plt.style.use('seaborn-v0_8')
    except:
        plt.style.use('ggplot')
    
    plt.rcParams['font.size'] = 12
    
    # Custom gradient colormap: blue (best) → orange → red (worst)
    cmap = LinearSegmentedColormap.from_list('ablation_cmap', ['#4e79a7', '#f28e2b', '#e15759'])
    
    # Bar positions
    x = np.arange(len(df))
    width = 0.35
    
    # Plot bars
    for i, model_name in enumerate(df.index):
        # Accuracy bar
        ax.bar(x[i] - width/2, df.loc[model_name, 'accuracy'], width,
               color=cmap(i / max(1, len(df)-1)), edgecolor='white', linewidth=1.2,
               label='Accuracy (%)' if i == 0 else "")
        
        # F2-Score bar (scaled to %)
        ax.bar(x[i] + width/2, df.loc[model_name, 'f2_score'] * 100, width,
               color=cmap(i / max(1, len(df)-1)), alpha=0.75, edgecolor='white', linewidth=1.2,
               label='F2-Score (scaled %)' if i == 0 else "")
    
    # Add value labels on bars
    for i, model_name in enumerate(df.index):
        ax.text(x[i] - width/2, df.loc[model_name, 'accuracy'] + 0.8,
                f"{df.loc[model_name, 'accuracy']:.2f}",
                ha='center', va='bottom', fontweight='bold', fontsize=10)
        
        ax.text(x[i] + width/2, df.loc[model_name, 'f2_score'] * 100 + 0.8,
                f"{df.loc[model_name, 'f2_score']:.4f}",
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Customize axes
    ax.set_xticks(x)
    ax.set_xticklabels([name.replace('_', ' ').title() for name in df.index],
                       fontsize=11, rotation=15, ha='right')
    ax.set_ylabel('Performance (%)', fontsize=12)
    ax.set_title('Ablation Study: Impact of Modalities on Severity Prediction',
                 fontsize=14, fontweight='bold', pad=20)
    
    ax.legend(loc='upper right', fontsize=10, framealpha=0.95)
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    
    # Set y-limit with some headroom
    max_val = max(df['accuracy'].max(), df['f2_score'].max() * 100)
    ax.set_ylim(0, max_val * 1.15)
    
    # Colorbar: left = best, right = worst
    sm = ScalarMappable(cmap=cmap, norm=plt.Normalize(0, max(1, len(df)-1)))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, ticks=[0, max(1, len(df)-1)], shrink=0.7, pad=0.02)
    cbar.ax.set_yticklabels(['Best', 'Worst'], fontsize=10)
    cbar.set_label('Performance Rank', rotation=270, labelpad=15)
    
    # Key insight annotation
    best_model = df.index[0].replace('_', ' ').title()
    best_acc = df['accuracy'].iloc[0]
    best_f2 = df['f2_score'].iloc[0]
    ax.annotate(f'Best: {best_model}\nAcc: {best_acc:.2f}% | F2: {best_f2:.4f}',
                xy=(0.02, 0.98), xycoords='axes fraction',
                ha='left', va='top', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='gray', alpha=0.9))
    
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, "images", 'ablation_results.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Ablation visualization saved as 'ablation_results.png'")

visualize_ablation_results(ablation_results)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from typing import Optional, Dict, Any
from transformers import DistilBertTokenizer
import os

# Load tokenizer (must be run before explainer)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

severity_map = {0: "Low", 1: "Medium", 2: "High"}

class MultimodalExplainer:
    def __init__(self, tokenizer=None):
        self.severity_labels = ["Low", "Medium", "High"]
        self.severity_colors = ['#4CAF50', '#FF9800', '#F44336']  # green, orange, red
        self.tokenizer = tokenizer  # optional, can be set later

    def predict_with_adjusted_threshold(self, fused_logits: torch.Tensor, high_sensitivity: float = 0.3) -> int:
        probabilities = F.softmax(fused_logits, dim=1)
        if probabilities[0, 2] > high_sensitivity:
            return 2  # High
        return probabilities.argmax(dim=1).item()

    @torch.no_grad()
    def explain_prediction(
        self,
        model,
        audio_input: torch.Tensor,   # [1, 40, T]
        text_input: torch.Tensor,    # [1, seq_len]
        true_severity: Optional[int] = None
    ) -> Dict[str, Any]:
        model.eval()
        device = next(model.parameters()).device
        
        audio_input = audio_input.to(device)
        text_input = text_input.to(device).long()
        
        # Forward pass
        fused_logits, audio_emb, text_emb = model(audio_input, text_input)
        
        probabilities = F.softmax(fused_logits, dim=1)[0]
        predicted_class = self.predict_with_adjusted_threshold(fused_logits)
        
        # Modality contribution
        audio_imp = audio_emb.norm(dim=1).mean().item()
        text_imp  = text_emb.norm(dim=1).mean().item()
        total = audio_imp + text_imp + 1e-8
        audio_imp /= total
        text_imp  /= total
        
        # Decode real text
        if self.tokenizer is None:
            real_text = "[Tokenizer not provided]"
        else:
            real_text = self.tokenizer.decode(text_input[0], skip_special_tokens=True).strip()
            if not real_text:
                real_text = "[Empty or padding-only input]"
        
        explanation = {
            'prediction': {
                'class': predicted_class,
                'class_name': self.severity_labels[predicted_class],
                'confidence': probabilities[predicted_class].item(),
                'probabilities': probabilities.cpu().numpy().tolist()
            },
            'modality_contribution': {
                'audio': float(audio_imp),
                'text': float(text_imp)
            },
            'decoded_text': real_text,
            'true_severity_name': severity_map.get(true_severity, "Unknown") if true_severity is not None else None,
            'measures': self.get_precautionary_measures(predicted_class)
        }
        
        return explanation

    def get_precautionary_measures(self, severity: int) -> list:
        base = {
            0: ["Log incident", "Continue monitoring"],
            1: ["Increase surveillance", "Alert security"],
            2: ["Immediate lockdown", "Alert police", "Evacuate if safe"]
        }
        return base.get(severity, [])

    def create_explanation_dashboard(
        self,
        model,
        audio_input,
        text_input,
        true_severity: Optional[int] = None,
        sample_idx=None
    ):
        explanation = self.explain_prediction(model, audio_input, text_input, true_severity)
        
        pred = explanation['prediction']
        contrib = explanation['modality_contribution']
        measures = explanation['measures']
        real_text = explanation['decoded_text']
        true_name = explanation.get('true_severity_name', 'N/A')
        
        fig = plt.figure(figsize=(18, 10))
        plt.suptitle(
            f"Real Sample Explanation Dashboard {f'- Sample {sample_idx}' if sample_idx else ''}\n"
            f"Predicted: {pred['class_name']} ({pred['confidence']:.1%}) | True: {true_name}",
            fontsize=16, fontweight='bold', y=0.98
        )
        
        # 1. Probability bar
        ax1 = plt.subplot2grid((3, 3), (0, 0))
        ax1.bar(self.severity_labels, pred['probabilities'], color=self.severity_colors)
        ax1.set_title("Severity Probabilities")
        ax1.set_ylim(0, 1)
        ax1.set_ylabel("Probability")
        
        # 2. Modality contribution
        ax2 = plt.subplot2grid((3, 3), (0, 1))
        ax2.bar(['Audio', 'Text'], [contrib['audio'], contrib['text']],
                color=['#2196F3', '#9C27B0'])
        ax2.set_title("Modality Contribution")
        ax2.set_ylim(0, 1)
        ax2.set_ylabel("Relative Importance")
        
        # 3. Precautionary measures
        ax3 = plt.subplot2grid((3, 3), (0, 2), rowspan=2)
        ax3.axis('off')
        ax3.text(0.02, 0.95, "Recommended Actions", fontsize=14, fontweight='bold')
        for i, m in enumerate(measures):
            ax3.text(0.05, 0.88 - i*0.07, f"• {m}", fontsize=11, va='top')
        
        # 4. Real decoded text + summary
        ax4 = plt.subplot2grid((3, 3), (1, 0), colspan=2)
        ax4.axis('off')
        summary = f"Predicted: {pred['class_name']} ({pred['confidence']:.1%})\n"
        if true_name != "Unknown":
            summary += f"Ground Truth: {true_name}\n"
        summary += f"\nDecoded Input Text:\n{real_text[:300]}..." if real_text else "No text decoded"
        
        ax4.text(0.02, 0.95, summary, fontsize=11, va='top',
                 bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.5'))
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        return fig

# Demo runner
def run_real_explanation_demo(explainer, model, test_loader, num_samples=3, show_plots=True, save_plots=True):
    print("Generating real-data explanation dashboards...\n")
    
    device = next(model.parameters()).device
    model.eval()
    
    save_folder = os.path.join(base_path, "images", "explanation_dashboards")
    os.makedirs(save_folder, exist_ok=True)
    print(f"Saving to: {os.path.abspath(save_folder)}\n")
    
    sample_count = 0
    for batch in test_loader:
        if sample_count >= num_samples:
            break
        
        audio, text, _, _, sev_a, sev_t = [t.to(device) for t in batch]
        
        sample_audio = audio[0:1]
        sample_text = text[0:1]
        true_sev = torch.max(sev_a[0], sev_t[0]).item()
        
        fig = explainer.create_explanation_dashboard(
            model,
            sample_audio,
            sample_text,
            true_severity=true_sev,
            sample_idx=sample_count + 1
        )
        
        if show_plots:
            try:
                plt.show()
                print(f"Displayed sample {sample_count+1} (true: {severity_map.get(true_sev, '?')})")
            except Exception as e:
                print(f"Display failed: {e}")
        
        if save_plots:
            save_path = os.path.join(save_folder, f'real_dashboard_sample_{sample_count+1}.png')
            try:
                fig.savefig(save_path, dpi=200, bbox_inches='tight')
                print(f"→ Saved: {save_path}")
            except Exception as e:
                print(f"Save failed: {e}")
        
        plt.close(fig)
        sample_count += 1
    
    print(f"\nProcessed {sample_count} real samples.")
    if save_plots:
        print(f"Files saved in: {os.path.abspath(save_folder)}")
        print("Open the folder to view PNGs.")

# Run it
explainer = MultimodalExplainer()  # no tokenizer needed in init
run_real_explanation_demo(explainer, model, test_loader, num_samples=3, show_plots=True, save_plots=True)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime

# Reuse your precautionary measures (no change)
def get_precautionary_measures(severity, audio_class=None, text_category=None):
    base = {
        0: ["Log incident", "Continue monitoring", "No immediate action"],
        1: ["Increase surveillance", "Alert security", "Notify management"],
        2: ["Immediate lockdown", "Alert police/emergency", "Evacuate if safe"]
    }
    measures = base.get(severity, [])
    if audio_class:
        measures.append(f"Audio trigger: {audio_class}")
    if text_category:
        measures.append(f"Text trigger: {text_category}")
    return measures[:5]  # limit for display


In [ ]:
def create_real_monitoring_dashboard(model, test_loader, num_incidents=5):
    device = next(model.parameters()).device
    model.eval()
    
    processed = []
    sample_indices = random.sample(range(len(test_loader.dataset)), min(num_incidents, len(test_loader.dataset)))
    
    with torch.no_grad():
        for idx in sample_indices:
            # Get single sample from dataset
            sample = test_loader.dataset[idx]
            audio = sample[0].unsqueeze(0).to(device)   # [1, 40, 100]
            text  = sample[1].unsqueeze(0).to(device)   # [1, seq_len]
            sev_a = sample[4]                           # scalar tensor
            sev_t = sample[5]
            true_sev = torch.max(sev_a, sev_t).item()
            
            # Forward
            fused_logits, audio_emb, text_emb = model(audio, text)
            probs = F.softmax(fused_logits, dim=1)[0].cpu().numpy()
            pred_sev = probs.argmax()
            conf = probs[pred_sev]
            
            # Decode text
            tokens = tokenizer.convert_ids_to_tokens(text[0].cpu().numpy())
            real_text = tokenizer.decode(text[0], skip_special_tokens=True).strip()[:120]
            if not real_text:
                real_text = "[No text content]"
            
            # Fake audio/text class for demo (replace if you have mapping)
            audio_class = "Unknown audio"
            text_category = "Unknown category"
            
            processed.append({
                'id': f"INC-{idx+1:03d}",
                'true_severity': true_sev,
                'pred_severity': pred_sev,
                'confidence': conf,
                'probs': probs,
                'text': real_text,
                'measures': get_precautionary_measures(pred_sev, audio_class, text_category)
            })
    
    if not processed:
        print("No samples processed.")
        return None
    
    # ── Dashboard Layout ────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Real-Time Monitoring Dashboard\n{len(processed)} Recent Incidents from Test Set", 
                 fontsize=16, fontweight='bold', y=0.98)
    
    # Severity summary bar
    ax1 = plt.subplot2grid((3, 2), (0, 0), colspan=2)
    counts = [0] * 3
    for inc in processed:
        counts[inc['pred_severity']] += 1
    colors = ['#4CAF50', '#FF9800', '#F44336']
    ax1.bar(['Low', 'Medium', 'High'], counts, color=colors)
    ax1.set_title("Predicted Severity Distribution")
    ax1.set_ylabel("Count")
    for i, c in enumerate(counts):
        ax1.text(i, c + 0.1, str(c), ha='center', fontsize=12)
    
    # High priority alerts
    ax2 = plt.subplot2grid((3, 2), (1, 0), colspan=2)
    ax2.axis('off')
    high_inc = [inc for inc in processed if inc['pred_severity'] == 2]
    high_inc = sorted(high_inc, key=lambda x: x['confidence'], reverse=True)[:3]
    
    ax2.text(0.5, 0.95, "High Severity Alerts (Prioritize)", ha='center', fontsize=14, color='red', fontweight='bold')
    y = 0.85
    for inc in high_inc:
        txt = f"• {inc['id']} | Conf: {inc['confidence']:.1%} | Text: {inc['text'][:60]}..."
        ax2.text(0.05, y, txt, fontsize=11, va='top')
        y -= 0.20
    
    # Detailed incidents table
    ax3 = plt.subplot2grid((3, 2), (2, 0), colspan=2)
    ax3.axis('off')
    ax3.text(0.5, 0.95, "Recent Incident Details", ha='center', fontsize=14, fontweight='bold')
    
    y = 0.88
    for inc in processed[:5]:
        color = ['#4CAF50', '#FF9800', '#F44336'][inc['pred_severity']]
        txt = f"{inc['id']} | Sev: {inc['pred_severity']} ({inc['confidence']:.1%}) | {inc['text'][:80]}..."
        ax3.text(0.05, y, txt, fontsize=11, color=color, va='top')
        y -= 0.12
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig

# Run it
print("Creating real monitoring dashboard from test set...")
fig = create_real_monitoring_dashboard(model, test_loader, num_incidents=5)
if fig is not None:
    fig.savefig(os.path.join(base_path, "images", "real_monitoring_dashboard.png"), dpi=200, bbox_inches='tight')
    plt.show()
    print("Dashboard saved as 'real_monitoring_dashboard.png'")

In [ ]:
!pip install dash

In [ ]:
pip install dash-bootstrap-components

In [ ]:
"""
app.py  –  Interactive Dash web dashboard for crime severity monitoring.

Run locally
-----------
    python app.py

Deploy on Render
----------------
    Start command : gunicorn app:server
    Build command : pip install -r requirements.txt
    Environment   : MODEL_PATH=models/best_model.pt  (optional override)
"""

import os
import random
from datetime import datetime
from pathlib import Path

import dash
import dash_bootstrap_components as dbc
from dash import dcc, html, Input, Output, State, callback
import plotly.express as px
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertModel

# ── Project root & model path ────────────────────────────────────────────────
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pt"


# ── Inline model definition (no external src/ needed) ────────────────────────
class MultimodalFusionModel(nn.Module):
    """
    Fuses a mel-spectrogram audio branch (CNN) with a DistilBERT text branch,
    then classifies crime severity into 3 classes: Low / Medium / High.
    """

    def __init__(self, num_classes: int = 3, text_hidden: int = 768,
                 audio_out: int = 128, fusion_hidden: int = 256):
        super().__init__()

        # ── Audio branch (1-D CNN over mel features) ─────────────────────────
        self.audio_cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.AdaptiveAvgPool1d(1),
        )
        self.audio_fc = nn.Linear(64, audio_out)

        # ── Text branch (DistilBERT) ──────────────────────────────────────────
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.text_fc = nn.Linear(text_hidden, audio_out)

        # ── Fusion head ───────────────────────────────────────────────────────
        self.fusion = nn.Sequential(
            nn.Linear(audio_out * 2, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(fusion_hidden, num_classes),
        )

    def forward(self, audio: torch.Tensor, input_ids: torch.Tensor,
                attention_mask: torch.Tensor | None = None):
        # audio : (B, T) or (B, 1, T)
        if audio.dim() == 2:
            audio = audio.unsqueeze(1)
        a = self.audio_cnn(audio).squeeze(-1)   # (B, 64)
        a = F.relu(self.audio_fc(a))             # (B, audio_out)

        # text
        bert_out = self.bert(input_ids=input_ids,
                             attention_mask=attention_mask)
        cls = bert_out.last_hidden_state[:, 0, :]  # (B, 768)
        t = F.relu(self.text_fc(cls))              # (B, audio_out)

        fused = torch.cat([a, t], dim=1)           # (B, audio_out*2)
        logits = self.fusion(fused)                # (B, num_classes)
        return logits, a, t


# ── Constants ─────────────────────────────────────────────────────────────────
SEVERITY_LABELS = {0: "Low", 1: "Medium", 2: "High"}

# Google Maps style: Green → Orange → Red
SEVERITY_COLORS = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}
MAP_COLORS      = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}

AUDIO_CLASSES = [
    "gun_shot", "siren", "drilling", "car_horn", "dog_bark",
    "jackhammer", "engine_idling", "street_music", "children_playing", "air_conditioner",
]

VIZAG_LOCATIONS = [
    {"name": "Downtown Visakhapatnam", "lat": 17.6868, "lon": 83.2185},
    {"name": "Industrial Zone",        "lat": 17.7333, "lon": 83.3167},
    {"name": "Residential Block",      "lat": 17.6667, "lon": 83.2000},
    {"name": "Beach Road",             "lat": 17.7200, "lon": 83.3300},
    {"name": "City Center",            "lat": 17.6900, "lon": 83.2200},
    {"name": "MVP Colony",             "lat": 17.7400, "lon": 83.3100},
    {"name": "Gajuwaka",               "lat": 17.6500, "lon": 83.2200},
    {"name": "Seethammadhara",         "lat": 17.7260, "lon": 83.3100},
    {"name": "Rushikonda",             "lat": 17.7800, "lon": 83.3800},
    {"name": "Madhurawada",            "lat": 17.8100, "lon": 83.3600},
]


# ── Model loading ──────────────────────────────────────────────────────────────
def load_model_and_tokenizer():
    device    = torch.device("cpu")
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model     = MultimodalFusionModel().to(device)
    if MODEL_PATH.exists():
        model.load_state_dict(
            torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
        )
        print(f"[App] Loaded model from {MODEL_PATH}")
    else:
        print("[App] WARNING – checkpoint not found at models/best_model.pt. Using demo data.")
    model.eval()
    return model, tokenizer, device


model, tokenizer, device = load_model_and_tokenizer()


# ── Helpers ────────────────────────────────────────────────────────────────────
def get_precautionary_measures(severity: int, audio_class: str = "Unknown") -> list:
    base = {
        0: ["Log incident", "Continue monitoring", "Routine check"],
        1: ["Increase surveillance", "Alert security team", "Notify management"],
        2: ["Immediate lockdown", "Alert police/emergency", "Evacuate if safe", "Activate full protocol"],
    }
    measures = list(base.get(severity, []))
    if audio_class not in ("Unknown", ""):
        measures.append(f"Audio trigger: {audio_class}")
    return measures


def incidents_to_df(incidents: list) -> pd.DataFrame:
    df = pd.DataFrame(incidents)
    if df.empty:
        return df
    df["timestamp"]      = pd.to_datetime(df["timestamp"])
    df["severity_label"] = df["severity"].map(SEVERITY_LABELS)
    df["color"]          = df["severity"].map(SEVERITY_COLORS)
    df["map_color"]      = df["severity"].map(MAP_COLORS)
    return df


def get_real_incidents(test_loader, num_samples: int = 10) -> list:
    """Pull predictions from a real DataLoader; falls back to demo on error."""
    incidents = []
    dataset_len = len(test_loader.dataset)
    if dataset_len == 0:
        print("Error: Test dataset is empty!")
        return []

    sample_indices = random.sample(range(dataset_len), min(num_samples, dataset_len))

    with torch.no_grad():
        for i, idx in enumerate(sample_indices):
            try:
                sample   = test_loader.dataset[idx]
                audio    = sample[0].unsqueeze(0).to(device)
                text_id  = sample[1].unsqueeze(0).to(device)
                sev_a    = sample[4].item()
                sev_t    = sample[5].item()
                true_sev = max(sev_a, sev_t)

                logits, _, _ = model(audio, text_id)
                probs    = F.softmax(logits, dim=1)[0].cpu().numpy()
                pred_sev = int(probs.argmax())
                conf     = float(probs[pred_sev])

                real_text = tokenizer.decode(text_id[0], skip_special_tokens=True).strip()[:120]
                if not real_text:
                    real_text = "[No text content]"

                audio_class = AUDIO_CLASSES[i % len(AUDIO_CLASSES)]
                loc         = VIZAG_LOCATIONS[i % len(VIZAG_LOCATIONS)]

                incidents.append({
                    "id":            f"Incident-{i + 1}",
                    "location":      loc["name"],
                    "lat":           loc["lat"],
                    "lon":           loc["lon"],
                    "timestamp":     (datetime.now() - pd.Timedelta(minutes=random.randint(0, 300))).strftime("%Y-%m-%d %H:%M:%S"),
                    "audio_class":   audio_class,
                    "text_category": "CRIME",
                    "text_content":  real_text,
                    "severity":      pred_sev,
                    "confidence":    conf,
                    "measures":      get_precautionary_measures(pred_sev, audio_class),
                })
            except Exception as e:
                print(f"Error on sample {idx}: {e}")
                continue

    return incidents


# ── Demo incidents (used when no test_loader is available) ────────────────────
DEMO_INCIDENTS = [
    {
        "id":            f"INC-{i:03d}",
        "location":      VIZAG_LOCATIONS[i % len(VIZAG_LOCATIONS)]["name"],
        "lat":           VIZAG_LOCATIONS[i % len(VIZAG_LOCATIONS)]["lat"],
        "lon":           VIZAG_LOCATIONS[i % len(VIZAG_LOCATIONS)]["lon"],
        "timestamp":     (datetime.now() - pd.Timedelta(minutes=i * 15)).strftime("%Y-%m-%d %H:%M:%S"),
        "audio_class":   AUDIO_CLASSES[i % len(AUDIO_CLASSES)],
        "text_category": "CRIME",
        "text_content":  f"Demo incident {i} – replace with real test_loader data.",
        "severity":      i % 3,
        "confidence":    round(0.75 + 0.05 * (i % 5), 2),
        "measures":      get_precautionary_measures(i % 3, AUDIO_CLASSES[i % len(AUDIO_CLASSES)]),
    }
    for i in range(10)
]

# Use real incidents if test_loader is defined in the calling scope, else demo
try:
    incidents = get_real_incidents(test_loader, num_samples=10)  # noqa: F821
    if not incidents:
        raise ValueError("Empty result from test_loader")
except Exception:
    incidents = DEMO_INCIDENTS

df = incidents_to_df(incidents)


# ── Figure factories ───────────────────────────────────────────────────────────
def create_severity_chart(data_df: pd.DataFrame):
    counts = (
        data_df["severity_label"]
        .value_counts()
        .reindex(["Low", "Medium", "High"], fill_value=0)
        .reset_index()
    )
    counts.columns = ["Severity", "Count"]
    fig = px.bar(counts, x="Severity", y="Count",
                 color="Severity",
                 color_discrete_map={"Low": "#4CAF50", "Medium": "#FF9800", "High": "#F44336"},
                 text="Count")
    fig.update_layout(
        title="Incident Severity Distribution",
        xaxis_title="Severity Level",
        yaxis_title="Number of Incidents",
        showlegend=False,
        paper_bgcolor="rgba(255,255,255,0.0)",
        plot_bgcolor="rgba(255,255,255,0.0)",
        font={"color": "#1a3a4a"},
    )
    fig.update_traces(textfont_color="#1a3a4a", textposition="outside")
    return fig


def create_map(data_df: pd.DataFrame):
    if data_df.empty:
        return px.scatter_mapbox(zoom=10, center={"lat": 17.6868, "lon": 83.2185},
                                 mapbox_style="open-street-map")
    fig = px.scatter_mapbox(
        data_df,
        lat="lat", lon="lon",
        color="severity_label",
        color_discrete_map={"Low": "#4CAF50", "Medium": "#FF9800", "High": "#F44336"},
        size=[16] * len(data_df),
        hover_name="id",
        hover_data=["location", "timestamp", "audio_class", "text_content", "confidence"],
        zoom=10,
        center={"lat": 17.6868, "lon": 83.2185},
        mapbox_style="open-street-map",
    )
    fig.update_layout(
        title="Geographic Distribution of Incidents",
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        legend_title="Severity", height=450,
        paper_bgcolor="rgba(0,0,0,0)",
        font={"color": "#1a3a4a"},
    )
    fig.update_traces(marker=dict(opacity=0.85))
    return fig


def create_timeseries(data_df: pd.DataFrame):
    if data_df.empty:
        return px.line(title="No data available")
    df2 = data_df.copy()
    df2["timestamp"] = pd.to_datetime(df2["timestamp"])
    df2["hour"]      = df2["timestamp"].dt.floor("h")
    counts = df2.groupby(["hour", "severity_label"]).size().reset_index(name="count")
    fig = px.line(
        counts, x="hour", y="count",
        color="severity_label",
        color_discrete_map={"Low": "#4CAF50", "Medium": "#FF9800", "High": "#F44336"},
        markers=True,
        title="Severity Trends Over Time",
    )
    fig.update_layout(
        paper_bgcolor="rgba(255,255,255,0.0)",
        plot_bgcolor="rgba(255,255,255,0.0)",
        font={"color": "#1a3a4a"},
        xaxis_title="Time",
        yaxis_title="Number of Incidents",
        legend_title="Severity",
    )
    return fig


# ── App layout ─────────────────────────────────────────────────────────────────
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.FLATLY],
    title="Crime Severity Monitor",
)
server = app.server  # for gunicorn

app.layout = dbc.Container([

    # ── Header ──────────────────────────────────────────────────────────────
    dbc.Row(dbc.Col(html.Div([
        html.H1("Multimodal Crime Severity Dashboard",
                className="text-center my-4", style={"color": "#17a2b8"}),
        html.P("Real-time detection using audio & text modalities | Locations are simulated for demo purposes",
               className="text-center text-muted mb-4"),
    ]))),

    # ── Confidence Threshold Slider ──────────────────────────────────────────
    dbc.Row(dbc.Col(dbc.Card(dbc.CardBody([
        html.Label("🎚️ Confidence Threshold Filter",
                   style={"color": "#1a3a4a", "fontWeight": "bold", "fontSize": "1.1em"}),
        dcc.Slider(
            id="confidence-slider",
            min=0.0, max=1.0, step=0.05,
            value=0.5,
            marks={0: "0%", 0.25: "25%", 0.5: "50%", 0.75: "75%", 1.0: "100%"},
            tooltip={"placement": "bottom", "always_visible": True},
        ),
        html.P("Only show incidents at or above this confidence level",
               style={"color": "#2a5a6a", "fontSize": "0.85em", "marginTop": "8px"}),
    ]), className="shadow mb-4"))),

    # ── Charts row ───────────────────────────────────────────────────────────
    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody(
            dcc.Graph(id="severity-chart")
        ), className="shadow"), width=6),
        dbc.Col(dbc.Card(dbc.CardBody(
            dcc.Graph(id="incident-map")
        ), className="shadow"), width=6),
    ], className="mb-4"),

    # ── Time-series chart ────────────────────────────────────────────────────
    dbc.Row(dbc.Col(dbc.Card(dbc.CardBody(
        dcc.Graph(id="timeseries-chart")
    ), className="shadow mb-4"))),

    # ── High severity alerts ─────────────────────────────────────────────────
    dbc.Row(dbc.Col(dbc.Card([
        dbc.CardHeader("⚠ High Severity Alerts", className="bg-danger text-white fw-bold"),
        dbc.CardBody(id="high-severity-list"),
    ], className="shadow mb-4"))),

    # ── Incidents table ──────────────────────────────────────────────────────
    dbc.Row(dbc.Col(dbc.Card([
        dbc.CardHeader("All Incidents", className="bg-primary text-white fw-bold"),
        dbc.CardBody(id="incidents-table"),
    ], className="shadow"))),

    # ── Stores & intervals ───────────────────────────────────────────────────
    dcc.Store(id="stored-incidents", data=incidents),
    dcc.Store(id="filtered-incidents", data=incidents),
    dcc.Interval(id="interval-component", interval=30_000, n_intervals=0),

    # ── Modal ────────────────────────────────────────────────────────────────
    dbc.Modal([
        dbc.ModalHeader(html.H4(id="modal-title"),
                        style={"background-color": "#1a6b8a", "color": "#ffffff"}),
        dbc.ModalBody(id="modal-content",
                      style={"background-color": "#f0f9fc", "color": "#1a3a4a"}),
        dbc.ModalFooter(
            dbc.Button("Close", id="close-modal", className="ms-auto", color="secondary")
        ),
    ], id="incident-modal", size="lg"),

], fluid=True, style={"background-color": "#e8f4f8", "min-height": "100vh",
                      "padding": "20px", "color": "#1a3a4a"})


# ── Callbacks ──────────────────────────────────────────────────────────────────

@callback(
    Output("filtered-incidents", "data"),
    Input("confidence-slider", "value"),
    State("stored-incidents", "data"),
)
def filter_by_confidence(threshold, all_incidents):
    return [inc for inc in all_incidents if inc["confidence"] >= threshold]


@callback(Output("severity-chart", "figure"), Input("filtered-incidents", "data"))
def update_severity_chart(filtered):
    return create_severity_chart(incidents_to_df(filtered))


@callback(Output("incident-map", "figure"), Input("filtered-incidents", "data"))
def update_map(filtered):
    return create_map(incidents_to_df(filtered))


@callback(Output("timeseries-chart", "figure"), Input("filtered-incidents", "data"))
def update_timeseries(filtered):
    return create_timeseries(incidents_to_df(filtered))


@callback(Output("high-severity-list", "children"), Input("filtered-incidents", "data"))
def update_high_severity(filtered):
    high = sorted(
        [inc for inc in filtered if inc["severity"] == 2],
        key=lambda x: x["confidence"], reverse=True,
    )
    if not high:
        return html.P("No high severity incidents above threshold.",
                      className="text-center p-3", style={"color": "#e0e0e0"})
    cards = []
    for inc in high:
        cards.append(dbc.Card(dbc.CardBody([
            html.H5(f"ID: {inc['id']} | Location: {inc['location']}",
                    style={"color": "#1a3a4a"}),
            html.P(f"Time: {inc['timestamp']} | Audio: {inc['audio_class']}",
                   style={"color": "#2a5a6a"}),
            html.P(inc["text_content"], style={"color": "#1a3a4a"}),
            html.Div([
                html.P("Recommended Measures:", className="mb-1",
                       style={"color": "#e65100", "fontWeight": "bold"}),
                html.Ul([html.Li(m, style={"color": "#1a3a4a"}) for m in inc["measures"][:4]]),
            ]),
            dbc.Button("View Details",
                       id={"type": "high-button", "index": inc["id"]},
                       color="danger", className="mt-2"),
        ], style={"background-color": "#fef9f9"}), className="mb-3 shadow border-0"))
    return cards


@callback(Output("incidents-table", "children"), Input("filtered-incidents", "data"))
def update_table(filtered):
    sorted_inc = sorted(filtered, key=lambda x: x["timestamp"], reverse=True)
    header = html.Thead(html.Tr([
        html.Th(col, style={"color": "#ffffff", "fontWeight": "bold"})
        for col in ["ID", "Location", "Time", "Audio", "Text", "Severity", "Confidence", "Action"]
    ], style={"background-color": "#1a6b8a"}))

    row_bg    = {0: "#e8f5e9", 1: "#fff3e0", 2: "#ffebee"}
    sev_badge = {0: "success", 1: "warning",  2: "danger"}

    rows = [
        html.Tr([
            html.Td(inc["id"],            style={"color": "#1a3a4a", "fontWeight": "500"}),
            html.Td(inc["location"],      style={"color": "#1a3a4a"}),
            html.Td(inc["timestamp"],     style={"color": "#2a5a6a", "fontSize": "0.9em"}),
            html.Td(inc["audio_class"],   style={"color": "#2a5a6a"}),
            html.Td(inc["text_category"], style={"color": "#2a5a6a"}),
            html.Td(html.Span(SEVERITY_LABELS[inc["severity"]],
                              className=f"badge bg-{sev_badge[inc['severity']]}",
                              style={"fontSize": "0.9em"})),
            html.Td(f"{inc['confidence']:.0%}", style={"color": "#1a3a4a"}),
            html.Td(dbc.Button("View Details",
                               id={"type": "table-button", "index": inc["id"]},
                               color="info", size="sm", outline=True)),
        ], style={"background-color": row_bg[inc["severity"]], "borderBottom": "1px solid #b0d4e0"})
        for inc in sorted_inc
    ]
    return dbc.Table([header, html.Tbody(rows)],
                     bordered=True, hover=True, responsive=True,
                     style={"borderColor": "#b0d4e0"})


def _modal_content(inc: dict):
    return html.Div([
        html.H5("Basic Information", style={"color": "#17a2b8"}),
        dbc.Row([
            dbc.Col(html.P(f"Location: {inc['location']}")),
            dbc.Col(html.P(f"Time: {inc['timestamp']}")),
        ]),
        dbc.Row([
            dbc.Col(html.P(f"Audio Class: {inc['audio_class']}")),
            dbc.Col(html.P(f"Text Category: {inc['text_category']}")),
        ]),
        html.H5("Description", style={"color": "#17a2b8"}),
        html.P(inc["text_content"]),
        html.H5("Severity Assessment", style={"color": "#17a2b8"}),
        dbc.Progress(
            value=inc["confidence"] * 100,
            color=["success", "warning", "danger"][inc["severity"]],
            striped=True, animated=True, style={"height": "25px"},
        ),
        html.P(f"Severity: {SEVERITY_LABELS[inc['severity']]} "
               f"(Confidence: {inc['confidence']:.1%})"),
        html.H5("Recommended Measures", style={"color": "#17a2b8"}),
        html.Ul([html.Li(m) for m in inc["measures"]]),
        html.H5("Location Map", style={"color": "#17a2b8"}),
        dcc.Graph(figure=px.scatter_mapbox(
            pd.DataFrame([inc]), lat="lat", lon="lon",
            zoom=14, mapbox_style="open-street-map",
        )),
    ], style={"color": "#1a3a4a"})


@callback(
    [Output("incident-modal", "is_open", allow_duplicate=True),
     Output("modal-title",    "children", allow_duplicate=True),
     Output("modal-content",  "children", allow_duplicate=True)],
    Input({"type": "high-button", "index": dash.ALL}, "n_clicks"),
    State("stored-incidents", "data"),
    State("incident-modal",   "is_open"),
    prevent_initial_call=True,
)
def open_modal_high(n_clicks, all_incidents, is_open):
    ctx = dash.callback_context
    if not ctx.triggered or not any(n_clicks):
        return dash.no_update, dash.no_update, dash.no_update
    inc_id = eval(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    inc    = next((i for i in all_incidents if i["id"] == inc_id), None)
    if inc:
        return True, f"Incident Details: {inc['id']}", _modal_content(inc)
    return is_open, dash.no_update, dash.no_update


@callback(
    [Output("incident-modal", "is_open"),
     Output("modal-title",    "children"),
     Output("modal-content",  "children")],
    Input({"type": "table-button", "index": dash.ALL}, "n_clicks"),
    State("stored-incidents", "data"),
    State("incident-modal",   "is_open"),
    prevent_initial_call=True,
)
def open_modal_table(n_clicks, all_incidents, is_open):
    ctx = dash.callback_context
    if not ctx.triggered or not any(n_clicks):
        return dash.no_update, dash.no_update, dash.no_update
    inc_id = eval(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    inc    = next((i for i in all_incidents if i["id"] == inc_id), None)
    if inc:
        return True, f"Incident Details: {inc['id']}", _modal_content(inc)
    return is_open, dash.no_update, dash.no_update


@callback(
    Output("incident-modal", "is_open", allow_duplicate=True),
    Input("close-modal", "n_clicks"),
    State("incident-modal", "is_open"),
    prevent_initial_call=True,
)
def close_modal(n_clicks, is_open):
    return not is_open if n_clicks else is_open


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app.run(debug=False, host="127.0.0.1", port=8050)